# Pareto sweep analysis

Load a `summary.csv` or `summary.json` from a `pareto_runs/` directory and visualize the Pareto front.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Point to a completed sweep output directory
RUN_DIR = Path("pareto_runs") / "dale_sanity3_5x5_2026-09-10_13-11-22"

csv_path = RUN_DIR / "summary.csv"
json_path = RUN_DIR / "summary.json"

df = pd.read_csv(csv_path)
with json_path.open() as f:
    meta = json.load(f)

print(f"Runs: {len(df)}  |  Pareto points: {df['is_pareto'].sum()}")
print(f"Tasks: {meta['active_tasks']}")
df.head()

: 

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

pareto = df["is_pareto"].astype(bool)
x = df["task_loss"].values
y = df["metabolic_cost"].values
z = df["wiring_cost"].values

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(x[~pareto], y[~pareto], z[~pareto], c="0.75", s=50, alpha=0.8, label="grid")
sc = ax.scatter(x[pareto], y[pareto], z[pareto], c="C1", s=90, depthshade=True, label="Pareto")

# Label Pareto points with (lambda_rate, lambda_connectivity)
for _, row in df.loc[pareto].iterrows():
    ax.text(
        row["task_loss"],
        row["metabolic_cost"],
        row["wiring_cost"],
        f"  ({row['lambda_rate']:.3g}, {row['lambda_connectivity']:.3g})",
        fontsize=8,
    )

ax.set_xlabel("task loss")
ax.set_ylabel("metabolic cost")
ax.set_zlabel("wiring cost")
ax.set_title(f"Pareto front — {meta.get('task_battery', '')} ({len(df)} runs)")
ax.legend(loc="upper left")
ax.view_init(elev=22, azim=-58)
fig.tight_layout()
plt.show()

In [ ]:
cols = ["lambda_rate", "lambda_connectivity", "task_loss", "metabolic_cost", "wiring_cost", "mean_acc", "is_pareto"]
df.sort_values(["task_loss", "metabolic_cost", "wiring_cost"])[cols]